# A-5b M 통합 실모델 확인 (개발 세트)

Colab **T4 GPU** 런타임에서 위에서부터 순서대로 실행한다.

- **개발 세트(`data/frozen/dev.jsonl`)만** 쓴다. 평가 세트로 프롬프트를 고치면 감점이다.
- M 파이프라인(`build_m`)은 `scripts.run_eval`이 만든다. 모델 태그·상한·예산·판정 형식(`judge.format`)은 모두 `configs/default.yaml`에서 읽는다.
- 로그는 문항마다 즉시 `results/runs/dev/r2/m.jsonl`에 쓴다. 셀이 끊겨도 같은 셀을 다시 실행하면 이어 돈다.
- 라운드는 **`r2`**를 쓴다. `r1`에는 B-4의 P1 개발 로그가 있고, 그 `RUN_META.json`의 설정 스냅샷에는 `judge` 섹션이 없어 이어 돌 수 없다.
- 실행 전에 로컬의 `YongMin` 브랜치가 GitHub에 push되어 있어야 한다.

| 볼 지표 | 기대 |
|---|---|
| 완주·오류 | 60문항 모두 기록, `error`·`timeout` 0 |
| 호출 수 | 문항당 4 이하, `budget_exhausted` 0 |
| 판정 잘림·파싱 실패 | 0에 가깝게 (C-4: 0%) |
| 정확도·호출 수·시간 | P1(`results/runs/dev/r1/p1.jsonl`)과 비교 |
| 기권률 | 답 있는 문항에서 과잉 기권이 크지 않은가 (C-4가 지적한 문제) |


## 0. GPU 확인

In [1]:
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

name, memory.used [MiB], memory.total [MiB]
Tesla T4, 0 MiB, 15360 MiB


## 1. 코드 받기 (`YongMin` 브랜치)

In [2]:
import os
os.chdir("/content")   # 다시 실행해도 repo/repo가 생기지 않게 항상 같은 위치에서 시작
REPO_URL = "https://github.com/minyi-k03/Intelligence-Software-Project.git"
# 비공개 저장소면 Colab 비밀(userdata)에 GH_TOKEN을 넣고 아래 두 줄 주석을 푼다. 토큰을 노트북에 적지 말 것.
# from google.colab import userdata
# REPO_URL = REPO_URL.replace("https://", f"https://{userdata.get('GH_TOKEN')}@")

if not os.path.isdir("repo"):
    !git clone {REPO_URL} repo
%cd /content/repo
!git fetch origin
!git checkout YongMin
!git pull --ff-only origin YongMin
!git log --oneline -3
assert os.path.isfile("tests/test_m_dev_smoke.py"), "A-5b 커밋이 없다. YongMin 브랜치가 push됐는지 확인할 것"


Cloning into 'repo'...
remote: Enumerating objects: 713, done.
remote: Counting objects: 100% (355/355), done.
remote: Compressing objects: 100% (200/200), done.
remote: Total 713 (delta 245), reused 238 (delta 155), pack-reused 358 (from 1)
Receiving objects: 100% (713/713), 790.35 KiB | 3.50 MiB/s, done.
Resolving deltas: 100% (398/398), done.
/content/repo
branch 'YongMin' set up to track 'origin/YongMin'.
Switched to a new branch 'YongMin'
From https://github.com/minyi-k03/Intelligence-Software-Project
 * branch            YongMin    -> FETCH_HEAD
Already up to date.
17e71c7 (HEAD -> YongMin, origin/YongMin) fix(notebook): r5 pull 실패로 M이 v3로 돈 사고 기록, 안전한 pull 확인 뒤 r6 재실행 셀
734bcbf feat(meba): M 구성 동결 - 판정 지시문 v2로 복원, 추가 실험 셀(r5 재측정·실패 단계·부트스트랩)
f666d83 docs(experiments): A-5b 판정 지시문 v1/v2/v3 dev 결과 - v2·v3 채택 기준 미달, 로그 r2~r4


## 2. 의존성

In [3]:
!pip install -q -r requirements.txt && pip list 2>/dev/null | grep -iE "^(langchain|langgraph|langchain-ollama) "

langchain                             1.4.8
langchain-ollama                      1.2.0
langgraph                             1.2.19


## 3. Ollama 설치와 서버 실행

In [4]:
# lspci/lshw가 없으면 설치 스크립트가 GPU를 못 찾는다 (Colab 이미지에는 기본으로 없음) -> 먼저 설치
!apt-get install -y -q zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, urllib.request
!pkill -f "ollama serve" || true      # 이전 실행에서 남은 서버가 있으면 끈다
time.sleep(2)

LOG = "/content/ollama.log"           # 서버 로그를 버리지 않고 남긴다
server = subprocess.Popen(["ollama", "serve"], stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
                          start_new_session=True)   # 셀을 중단(Ctrl-C)해도 서버가 같이 죽지 않게 한다
for _ in range(60):                   # 응답할 때까지 최대 60초 기다린다
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        print("서버 준비됨")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(f"Ollama 서버가 60초 안에 뜨지 않았다. 서버 로그 끝: {open(LOG).read()[-2000:]}")
!ollama --version
print("--- 서버 로그에서 GPU 관련 줄")
!grep -iE "cuda|gpu|inference compute|library" {LOG} | head -10

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
^C
서버 준비됨
ollama version is 0.40.18
--- 서버 로그에서 GPU 관련 줄
time=2026-11-26T04:17:17.224Z level=INFO source=routes.go:2484 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_CREATE_REMOTE:false OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0

## 4. 모델 받기 (태그는 `configs/default.yaml`에서 읽는다)

In [5]:
import subprocess, yaml
tag = yaml.safe_load(open("configs/default.yaml", encoding="utf-8"))["model"]["tag"]
print("model.tag =", tag)
r = subprocess.run(["ollama", "pull", tag], capture_output=True, text=True)   # 진행 막대는 버리고 결과만 본다
print("pull 종료 코드:", r.returncode, "|", r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-300:])
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

model.tag = qwen2.5:7b-instruct-q4_K_M
pull 종료 코드: 0 | 0% ▕██████████████████▏ 4.7 GB                         
pulling 832dd9e00a68:  72% ▕████████████      ▏ 8.2 KB/ 11 KB                  
pulling 2f15b3218f05: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 

NAME                          ID              SIZE      MODIFIED               
qwen2.5:7b-instruct-q4_K_M    845dbda0ea48    4.7 GB    Less than a second ago    



## 4-1. 모델이 GPU로 올라갔는지 확인
`PROCESSOR`가 `100% GPU`여야 한다. `CPU`가 보이면 멈추고 확인한다 (지연 시간이 크게 늘어난다).

In [6]:
import subprocess, yaml
tag = yaml.safe_load(open("configs/default.yaml", encoding="utf-8"))["model"]["tag"]
print(subprocess.run(["ollama", "run", tag, "Say OK."], capture_output=True, text=True, timeout=300).stdout)
ps = subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout
print(ps)
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
assert "GPU" in ps and "CPU" not in ps.replace("PROCESSOR", ""), "모델이 GPU로 올라가지 않았다. 위 서버 로그와 ollama ps를 확인할 것"
print("OK: GPU로 로드됨")

OK.


NAME                          ID              SIZE      PROCESSOR    CONTEXT    RUNNER      UNTIL              
qwen2.5:7b-instruct-q4_K_M    845dbda0ea48    4.8 GB    100% GPU     4096       llamacpp    4 minutes from now    

memory.used [MiB], memory.total [MiB]
4663 MiB, 15360 MiB
OK: GPU로 로드됨


## 5. 단위 테스트 (GPU 없이 도는 테스트)

가짜 모델 스모크(`test_m_dev_smoke.py`)가 실제 코퍼스·개발 세트 60문항으로 로그 규격을 이미 확인한다. 여기서는 환경이 같은지만 본다.

In [7]:
!python -m pytest tests/test_graph.py tests/test_judge.py tests/test_requery.py tests/test_config.py tests/test_m_dev_smoke.py -q 2>&1 | tail -3

........................................................................ [ 61%]
..............................................                           [100%]
118 passed in 4.34s


## 6. 동결 해시 확인
해시가 `data/README.md` 표와 다르면 여기서 멈춘다 (`run_eval`도 시작 전에 다시 확인한다).

In [8]:
import json
from collections import Counter
from pathlib import Path

from src.eval.logger import verify_frozen_hashes

hashes = verify_frozen_hashes(Path("data/frozen"), Path("data/README.md"))
print({name: digest[:12] for name, digest in hashes.items()})
items = [json.loads(line) for line in Path("data/frozen/dev.jsonl").open(encoding="utf-8")]
print(len(items), "문항", Counter(item["type"] for item in items))
!grep -n "format" configs/default.yaml


{'eval.jsonl': '44de0d8a6cf7', 'dev.jsonl': '8f18f76dd903', 'corpus.jsonl': 'd562adc17741'}
60 문항 Counter({'hotpotqa': 20, 'squad2': 20, 'gsm8k': 20})
28:  format: "json"                    # Evidence-judgment output format: "json" (ADR-0004, confirmed in Experiment C-4) or "tag" (control)


## 7. 검증 모드 (앞 3문항, `_scratch` 묶음)

실모델이 한 문항이라도 끝까지 도는지, 로그가 규격대로 나오는지 먼저 본다. 이 결과는 git 제외 폴더에 쓴다.

In [9]:
!python -m scripts.run_eval --verify 3 --pipelines m
rows = [json.loads(l) for l in Path("results/runs/_scratch/r1/m.jsonl").open(encoding="utf-8")]
for r in rows:
    print(r["qid"], "|", r["route"], r["decisions"], "| 답:", repr(r["answer"]), "| 정답:", r["gold"], "| calls:", r["n_calls"], "| error:", r["error"])
print(sorted(rows[0]))


[m r1]   1 hotpotqa-0040 wrong 13.4s
[m r1]   2 hotpotqa-0041 ok 2.2s
[m r1]   3 hotpotqa-0042 wrong 4.0s
완료: _scratch/r1 최대 VRAM(MiB) {'m': 5061}
hotpotqa-0040 | search ['RETRIEVE', 'RETRIEVE', 'ABSTAIN'] | 답: 'UNANSWERABLE' | 정답: ['Aristotle'] | calls: 4 | error: None
hotpotqa-0041 | search ['ANSWER'] | 답: 'United States Auto Club' | 정답: ['United States Auto Club'] | calls: 2 | error: None
hotpotqa-0042 | search ['RETRIEVE', 'ANSWER'] | 답: 'Reinvestment risk' | 정답: ['terrorist activity'] | calls: 3 | error: None
['abstained', 'answer', 'answer_kind', 'budget_exhausted', 'bundle', 'calls', 'correct', 'decisions', 'em', 'error', 'evidence_kept', 'f1', 'gold', 'is_warmup', 'kept_windows', 'latency_s', 'loop_breaks', 'max_input_tokens', 'n_calls', 'n_retries', 'numeric_correct', 'order_idx', 'pipeline', 'qid', 'retrieved_doc_ids', 'round', 'route', 'session_idx', 'subtype', 'timeout', 'type']


## 8. 개발 세트 60문항 전체 실행

문항마다 즉시 저장한다. 끊기면 같은 셀을 다시 실행하면 이어 돈다. 설정을 바꿨다면 `r3`처럼 새 라운드로 돌려야 한다 (한 라운드에 조건이 섞이면 러너가 멈춘다).

In [10]:
!python -m scripts.run_eval --bundle dev --round r2 --pipelines m

[m r2]   1 hotpotqa-0040 wrong 7.4s
[m r2]   2 hotpotqa-0041 ok 2.2s
[m r2]   3 hotpotqa-0042 wrong 4.2s
[m r2]   4 hotpotqa-0043 wrong 8.0s
[m r2]   5 hotpotqa-0044 wrong 7.9s
[m r2]   6 hotpotqa-0045 wrong 9.2s
[m r2]   7 hotpotqa-0046 ok 4.9s
[m r2]   8 hotpotqa-0047 wrong 3.6s
[m r2]   9 hotpotqa-0048 wrong 9.2s
[m r2]  10 hotpotqa-0049 wrong 2.7s
[m r2]  11 hotpotqa-0050 wrong 4.6s
[m r2]  12 hotpotqa-0051 wrong 8.8s
[m r2]  13 hotpotqa-0052 ok 2.1s
[m r2]  14 hotpotqa-0053 wrong 7.8s
[m r2]  15 hotpotqa-0054 wrong 8.1s
[m r2]  16 hotpotqa-0055 wrong 2.9s
[m r2]  17 hotpotqa-0056 wrong 2.6s
[m r2]  18 hotpotqa-0057 wrong 2.4s
[m r2]  19 hotpotqa-0058 wrong 2.5s
[m r2]  20 hotpotqa-0059 wrong 2.8s
[m r2]  21 squad2-0040 wrong 7.8s
[m r2]  22 squad2-0041 wrong 2.1s
[m r2]  23 squad2-0042 wrong 2.4s
[m r2]  24 squad2-0043 ok 5.7s
[m r2]  25 squad2-0044 wrong 8.4s
[m r2]  26 squad2-0045 wrong 8.1s
[m r2]  27 squad2-0046 wrong 7.2s
[m r2]  28 squad2-0047 wrong 7.9s
[m r2]  29 squad2-00

## 9. 요약표

집계 규칙은 `EVALUATION §4`·`INTERFACES §7.2`를 따른다.

- **정확도·기권·잘림·파싱 지표**: 워밍업 문항을 **포함한 60문항 전체**
- **비용·시간 지표**(`n_calls`, `out_tokens`, `latency_s`): 파이프라인의 첫 1문항(워밍업)을 **제외한 59문항**

| 열 | 뜻 |
|---|---|
| `retries` | 잘림 재시도 횟수 합 |
| `judge_trunc_first` | 판정 호출의 **1차 응답**이 출력 상한에 걸린 문항 비율 (재시도로 복구된 것 포함) |
| `judge_trunc` | **재시도 뒤에도** 잘린 판정 호출이 있는 문항 비율 (`scripts/compare_judge_format.py`의 `truncated_after_retry`와 같은 기준) |
| `parse_fail` | 판정 파싱 실패가 있는 문항 비율 |
| `over_abstain` | 답이 있는 문항(`answer_kind != unanswerable`) 중 기권한 비율 |


In [11]:
import pandas as pd

OUT = Path("results/runs/dev/r2/m.jsonl")
df = pd.DataFrame([json.loads(l) for l in OUT.open(encoding="utf-8")])
df["group"] = df.apply(lambda r: f"{r['type']}-{r['subtype']}" if r["type"] == "squad2" else r["type"], axis=1)
judge_calls = df["calls"].map(lambda cs: [c for c in cs if c["step"] == "judge"])
df["judge_trunc_first"] = judge_calls.map(lambda cs: any(c["truncated"] and not c["is_retry"] for c in cs))
df["judge_trunc"] = judge_calls.map(lambda cs: any(c["truncated"] and c["is_retry"] for c in cs))
df["parse_fail"] = judge_calls.map(lambda cs: any(c["parse_ok"] is False for c in cs))
df["total_out"] = df["calls"].map(lambda cs: sum(c["output_tokens"] for c in cs))
df["over_abstain"] = df["abstained"] & (df["answer_kind"] != "unanswerable")
cost = df[~df["is_warmup"]]   # 비용·시간 집계만 워밍업 제외. 정확도·부작용 지표는 전체(df)

print("완주:", len(df), "/ 60 | error:", int(df["error"].notna().sum()), "| timeout:", int(df["timeout"].sum()),
      "| budget_exhausted:", int(df["budget_exhausted"].sum()))
quality = df.groupby("group").agg(n=("qid", "count"), correct=("correct", "mean"), abstained=("abstained", "mean"),
                                  retries=("n_retries", "sum"), judge_trunc_first=("judge_trunc_first", "mean"),
                                  judge_trunc=("judge_trunc", "mean"), parse_fail=("parse_fail", "mean"))
spend = cost.groupby("group").agg(n_calls=("n_calls", "mean"), out_tokens=("total_out", "mean"),
                                  latency_s=("latency_s", "mean"))
pd.set_option("display.width", 200)
print(quality.join(spend).round(3))
print(f"전체 {len(df)}문항 | 정확도:", round(df["correct"].mean(), 3), "| 과잉 기권:",
      round(df["over_abstain"].sum() / (df["answer_kind"] != "unanswerable").sum(), 3))
print(f"워밍업 제외 {len(cost)}문항 | 평균 호출:", round(cost["n_calls"].mean(), 2),
      "| 평균 출력 토큰:", round(cost["total_out"].mean(), 1), "| 평균 시간:", round(cost["latency_s"].mean(), 2), "s")
print("경로:", dict(Counter(df["route"])))
print("판정:", dict(Counter(d for ds in df["decisions"] for d in ds)))
print("evidence_kept(HotpotQA):", df["evidence_kept"].dropna().mean())


완주: 60 / 60 | error: 0 | timeout: 0 | budget_exhausted: 0
                      n  correct  abstained  retries  judge_trunc_first  judge_trunc  parse_fail  n_calls  out_tokens  latency_s
group                                                                                                                           
gsm8k                20     0.85       0.00        2                0.0          0.0         0.0    1.300     239.150      7.837
hotpotqa             20     0.15       0.65        0                0.0          0.0         0.0    2.895     129.368      5.081
squad2-answerable    10     0.20       0.60        0                0.0          0.0         0.0    3.100     134.200      5.399
squad2-unanswerable  10     0.70       0.70        0                0.0          0.0         0.0    2.500      94.700      3.565
전체 60문항 | 정확도: 0.483 | 과잉 기권: 0.38
워밍업 제외 59문항 | 평균 호출: 2.32 | 평균 출력 토큰: 161.5 | 평균 시간: 5.81 s
경로: {'search': 40, 'math': 20}
판정: {'RETRIEVE': 35, 'ABSTAIN': 26, 'ANSWE

## 10. P1과 비교 (같은 개발 세트, `r1`의 P1 로그)

정확도·정답 겹침은 60문항 전체, 호출 수·시간은 **양쪽 모두** 워밍업이 아닌 문항만 비교한다.


In [12]:
P1 = Path("results/runs/dev/r1/p1.jsonl")
p1 = pd.DataFrame([json.loads(l) for l in P1.open(encoding="utf-8")])
both = df.merge(p1[["qid", "correct", "n_calls", "latency_s", "is_warmup"]], on="qid", suffixes=("_m", "_p1"))
spent = both[~both["is_warmup_m"] & ~both["is_warmup_p1"]]   # 비용·시간 비교는 워밍업 제외
print(f"문항: {len(both)} (호출·시간 비교: {len(spent)}문항, 워밍업 제외)")
print("정확도  M:", round(both["correct_m"].mean(), 3), " P1:", round(both["correct_p1"].mean(), 3))
print("호출    M:", round(spent["n_calls_m"].mean(), 2), " P1:", round(spent["n_calls_p1"].mean(), 2))
print("시간    M:", round(spent["latency_s_m"].mean(), 2), " P1:", round(spent["latency_s_p1"].mean(), 2))
print("M만 정답:", int((both["correct_m"] & ~both["correct_p1"]).sum()), " P1만 정답:", int((~both["correct_m"] & both["correct_p1"]).sum()))


문항: 60 (호출·시간 비교: 59문항, 워밍업 제외)
정확도  M: 0.483  P1: 0.35
호출    M: 2.32  P1: 1.75
시간    M: 5.81  P1: 9.02
M만 정답: 14  P1만 정답: 6


## 11. 이상 문항 (원문 확인)

오류·시간 초과·판정 잘림(1차 포함)·파싱 실패·예산 소진 문항만 본다. 1차만 잘리고 재시도로 복구된 문항도 여기 나온다.


In [13]:
odd = df[df["error"].notna() | df["timeout"] | df["judge_trunc_first"] | df["judge_trunc"] | df["parse_fail"]
         | df["budget_exhausted"]]
print(f"{len(odd)}문항")
for _, r in odd.iterrows():
    print("=" * 100)
    print(f"{r['qid']} route={r['route']} decisions={r['decisions']} answer={r['answer']!r} error={r['error']}")
    print("  calls (step, out_tokens, truncated, is_retry, parse_ok):",
          [(c["step"], c["output_tokens"], c["truncated"], c["is_retry"], c["parse_ok"]) for c in r["calls"]])


0문항


## 12. 결과 내려받기

Colab 런타임이 끝나면 파일은 사라진다. 로그(`results/runs/dev/r2/`)를 내려받아 레포에 올리고, 해석은 `docs/experiments/`에 남긴다.

In [16]:
!cd results/runs/dev && zip -qr /content/a5b_dev_r2.zip r2
from google.colab import files
files.download("/content/a5b_dev_r2.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 13. 판정 진단: 근거에 정답이 있었는데 RETRIEVE·ABSTAIN했나 (r2)

r2 결과에서 HotpotQA 기권 65%, SQuAD 답 있음 기권 60%, 과잉 기권 0.38이 나왔다. 그런데 `evidence_kept`(HotpotQA 근거 문장 보존율)는 0.9다. 근거는 모였는데 판정이 답을 확정하지 못한다는 뜻인지 확인한다.

- 로그의 `retrieved_doc_ids`·`kept_windows`로 판정 호출이 본 압축 근거를 회차별로 복원한다 (모델 호출 없음).
- "근거에 정답 있음" = 채점과 같은 정규화를 거친 근거 텍스트에 정답 문자열이 들어 있음 (C-4와 같은 기준). 정답 문자열이 있어도 그 문장만으로 답할 수 있다는 보장은 아니다.
- 대상: 검색 경로를 탄, 답이 있는 문항


In [9]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd

from src.config.settings import get_settings
from src.eval.scorer import _normalize          # 채점과 같은 정규화
from src.meba.compress import build_windows, render

ITEMS = {d["qid"]: d for d in map(json.loads, open("data/frozen/dev.jsonl", encoding="utf-8"))}
CORPUS = {d["doc_id"]: d for d in map(json.loads, open("data/frozen/corpus.jsonl", encoding="utf-8"))}
WINDOW = get_settings().compress.window


def load_round(rnd):
    """라운드의 M 로그를 DataFrame으로 읽는다."""
    rows = [json.loads(l) for l in Path(f"results/runs/dev/{rnd}/m.jsonl").open(encoding="utf-8")]
    df = pd.DataFrame(rows)
    df["group"] = df.apply(lambda r: f"{r['type']}-{r['subtype']}" if r["type"] == "squad2" else r["type"], axis=1)
    return df


def evidence_by_round(row):
    """판정 회차별 압축 근거 텍스트 (판정은 앞 회차 근거에 이어 붙인 것을 본다)."""
    out = []
    for doc_ids, kept in zip(row["retrieved_doc_ids"], row["kept_windows"]):
        paras = [CORPUS[i] for i in doc_ids]
        by_id = {w.window_id: w for w in build_windows(paras, WINDOW)}
        out.append(render(paras, [by_id[k] for k in kept if k in by_id]))
    return out


def diagnose(df):
    """검색 경로 · 답 있는 문항마다 결과와 '근거에 정답 있음'을 붙인 표."""
    recs = []
    for _, r in df[(df["route"] == "search") & (df["answer_kind"] != "unanswerable")].iterrows():
        golds = [_normalize(g) for g in ITEMS[r["qid"]]["answers"]]
        ev = [_normalize(e) for e in evidence_by_round(r)]
        outcome = "정답" if r["correct"] else ("기권" if r["abstained"] else "오답")
        recs.append({"qid": r["qid"], "group": r["group"], "outcome": outcome,
                     "decisions": "→".join(r["decisions"]), "n_calls": r["n_calls"],
                     "gold_in_first": bool(ev) and any(g and g in ev[0] for g in golds),
                     "gold_in_any": any(g and g in e for g in golds for e in ev)})
    return pd.DataFrame(recs)


def show_diagnosis(rnd):
    d = diagnose(load_round(rnd))
    print(f"[{rnd}] 검색 경로 · 답 있는 문항 {len(d)}개")
    print(pd.crosstab(d["outcome"], d["gold_in_any"].map({True: "근거에 정답 있음", False: "없음"}), margins=True))
    first_hit = d[d["gold_in_first"]]
    print(f"\n1차 근거에 이미 정답이 있던 {len(first_hit)}문항의 첫 판정:",
          dict(Counter(s.split("→")[0] for s in first_hit["decisions"])))
    return d


diag_r2 = show_diagnosis("r2")
print()
print(diag_r2[(diag_r2["outcome"] != "정답") & diag_r2["gold_in_any"]].to_string(index=False))


[r2] 검색 경로 · 답 있는 문항 30개
gold_in_any  근거에 정답 있음  없음  All
outcome                        
기권                  14   5   19
오답                   5   1    6
정답                   5   0    5
All                 24   6   30

1차 근거에 이미 정답이 있던 24문항의 첫 판정: {'RETRIEVE': 12, 'ANSWER': 7, 'ABSTAIN': 5}

          qid             group outcome                 decisions  n_calls  gold_in_first  gold_in_any
hotpotqa-0040          hotpotqa      기권 RETRIEVE→RETRIEVE→ABSTAIN        4           True         True
hotpotqa-0044          hotpotqa      기권 RETRIEVE→RETRIEVE→ABSTAIN        4           True         True
hotpotqa-0048          hotpotqa      기권 RETRIEVE→RETRIEVE→ABSTAIN        4           True         True
hotpotqa-0049          hotpotqa      오답                    ANSWER        2           True         True
hotpotqa-0051          hotpotqa      기권 RETRIEVE→RETRIEVE→ABSTAIN        4           True         True
hotpotqa-0053          hotpotqa      기권 RETRIEVE→RETRIEVE→ABSTAIN        4           True 

## 14. 판정 프롬프트 v2로 다시 실행 (r3)

13의 진단에서 "근거에 정답이 있는데 기권·재검색"이 많으면 판정 프롬프트를 고친다 (C-4 후속 작업과 같은 방향). v2에서 바꾼 것은 `src/meba/judge.py`의 지시문 세 군데다. **ABSTAIN 정의와 공통 답 형식 지시문(`CONTENT_INSTRUCTION`)은 그대로** 두었다.

| 자리 | v1 | v2 |
|---|---|---|
| ANSWER | the evidence states the answer. | 한 문장에 있거나 여러 문단을 합쳐 답을 댈 수 있으면, 질문과 표현이 달라도 ANSWER |
| RETRIEVE | 근거가 질문과 관련 있지만 필요한 사실 하나가 없음 | **아직 답을 댈 수 없고**, 필요한 사실 하나가 없음 |
| 빠진 정보 | 사람·장소·작품·용어를 적은 빠진 사실 | 같음 + 답 후보를 쓰지 말 것 |
| 서술(reasoning) | 근거가 답하는지 한두 문장 | 답이 되는 근거 부분을 **인용**하거나, 빠진 사실을 말함 |

- **반드시 새 라운드 `r3`로 돈다.** `RUN_META`는 설정 스냅샷만 비교하고 git 커밋은 기록만 하므로, 코드를 pull한 뒤 `r2`를 이어 돌리면 막히지 않고 v1·v2가 섞인다.
- 수학 경로(라우터)는 바뀌지 않았다. temperature 0이므로 GSM8K 결과가 r2와 같아야 한다 (15에서 확인).


In [18]:
assert Path("results/runs/dev/r2/m.jsonl").exists(), "비교 기준 r2 로그가 없다 (런타임이 초기화됐으면 r2 zip을 먼저 올릴 것)"
!git pull --ff-only origin YongMin
!git log --oneline -1
!grep -n "quote the part of the evidence" src/meba/judge.py
!python -m pytest tests/test_judge.py tests/test_m_dev_smoke.py -q 2>&1 | tail -1
!python -m scripts.run_eval --bundle dev --round r3 --pipelines m


remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 13 (delta 10), reused 13 (delta 10), pack-reused 0 (from 0)
Unpacking objects: 100% (13/13), 19.98 KiB | 2.50 MiB/s, done.
From https://github.com/minyi-k03/Intelligence-Software-Project
 * branch            YongMin    -> FETCH_HEAD
   69b824e..9d8f0a4  YongMin    -> origin/YongMin
Updating 69b824e..9d8f0a4
Fast-forward
 .../experiments/2026-11-26-a-m-dev-judge-prompt.md | 109 +++++
 docs/progress.md                                   |   2 +-
 notebooks/verify_a5b.ipynb                         | 519 +++++++++++++++++++--
 src/meba/judge.py                                  |  24 +-
 tests/test_judge.py                                |  19 +
 5 files changed, 640 insertions(+), 33 deletions(-)
 create mode 100644 docs/experiments/2026-11-26-a-m-dev-judge-prompt.md
9d8f0a4 (HEAD -> YongMin, origin/YongMin) feat(meba): A-5b dev 실행 진단 반영 - 근거 판정 지

## 15. r2(v1) vs r3(v2) 비교

채택 기준은 미리 정한다. SPEC §6의 조건부 부품 기준을 그대로 쓴다: **유형별 정답 수가 줄지 않고 목표 지표가 좋아지면 채택.**

- 목표 지표: 과잉 기권률, 문항당 호출 수 (워밍업 제외)
- 따로 볼 것: SQuAD 답할 수 없음의 정답 수 (기권을 덜 하게 만들면 여기가 먼저 나빠진다)
- 문항별 정답→오답 변화는 목록으로 남긴다. McNemar 정확 검정은 60문항이라 참고용이다.


In [10]:
import math


def mcnemar_exact(b, c):
    """양측 정확 McNemar p값 (불일치 쌍 b, c)."""
    n = b + c
    if n == 0:
        return 1.0
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(min(b, c) + 1)) / 2 ** n)


def round_summary(df):
    cost = df[~df["is_warmup"]]
    q = df.groupby("group").agg(n=("qid", "count"), correct=("correct", "sum"), abstained=("abstained", "sum"))
    s = cost.groupby("group").agg(n_calls=("n_calls", "mean"), latency_s=("latency_s", "mean"))
    return q.join(s)


v1, v2 = load_round("r2"), load_round("r3")
print(pd.concat({"v1 (r2)": round_summary(v1), "v2 (r3)": round_summary(v2)}, axis=1).round(2).to_string())

answerable = lambda d: (d["answer_kind"] != "unanswerable").sum()
for name, d in (("v1 (r2)", v1), ("v2 (r3)", v2)):
    c = d[~d["is_warmup"]]
    print(f"{name}: 정답 {int(d['correct'].sum())}/60 | 과잉 기권 {(d['abstained'] & (d['answer_kind'] != 'unanswerable')).sum() / answerable(d):.2f}"
          f" | 호출 {c['n_calls'].mean():.2f} | 시간 {c['latency_s'].mean():.2f}s | 오류 {int(d['error'].notna().sum())}"
          f" | 예산 소진 {int(d['budget_exhausted'].sum())} | 판정 {dict(Counter(x for xs in d['decisions'] for x in xs))}")

both = v1.merge(v2[["qid", "correct", "answer", "decisions", "route"]], on="qid", suffixes=("_v1", "_v2"))
gained = both[~both["correct_v1"] & both["correct_v2"]]
lost = both[both["correct_v1"] & ~both["correct_v2"]]
print(f"\nv2만 정답 {len(gained)} / v1만 정답 {len(lost)} | McNemar 정확 p = {mcnemar_exact(len(gained), len(lost)):.3f}")
for title, part in (("v2에서 맞힌 문항", gained), ("v2에서 틀린 문항 (정답→오답)", lost)):
    print(f"\n--- {title}")
    for _, r in part.iterrows():
        print(f"{r['qid']:15s} {r['decisions_v1']} {r['answer_v1']!r} → {r['decisions_v2']} {r['answer_v2']!r} | 정답 {ITEMS[r['qid']]['answers']}")

math_rows = both[both["route_v1"] == "math"]
same_math = (math_rows["answer_v1"] == math_rows["answer_v2"]).all() and (math_rows["route_v2"] == "math").all()
print("\n수학 경로 결과가 r2와 같은가:", same_math)

# P1(r1)과 비교: 시간 초과 문항은 호출 기록이 없고 시간은 120초로 남는다 (EVALUATION §4) -> 함께 보고한다.
p1 = pd.DataFrame([json.loads(l) for l in Path("results/runs/dev/r1/p1.jsonl").open(encoding="utf-8")])
p1c = p1[~p1["is_warmup"]]
v2c = v2[~v2["is_warmup"]]
print(f"\nP1(r1): 정답 {int(p1['correct'].sum())}/60 | 호출 {p1c['n_calls'].mean():.2f} | 시간 평균 {p1c['latency_s'].mean():.2f}s"
      f" · 중앙값 {p1c['latency_s'].median():.2f}s | 시간 초과 {int(p1['timeout'].sum())}")
print(f"M v2(r3): 정답 {int(v2['correct'].sum())}/60 | 호출 {v2c['n_calls'].mean():.2f} | 시간 평균 {v2c['latency_s'].mean():.2f}s"
      f" · 중앙값 {v2c['latency_s'].median():.2f}s | 시간 초과 {int(v2['timeout'].sum())}")


                    v1 (r2)                                     v2 (r3)                                    
                          n correct abstained n_calls latency_s       n correct abstained n_calls latency_s
group                                                                                                      
gsm8k                    20      17         0    1.30      7.84      20      17         0    1.30      7.78
hotpotqa                 20       3        13    2.89      5.08      20       5         8    2.26      3.70
squad2-answerable        10       2         6    3.10      5.40      10       5         1    2.30      3.37
squad2-unanswerable      10       7         7    2.50      3.57      10       4         4    2.00      2.56
v1 (r2): 정답 29/60 | 과잉 기권 0.38 | 호출 2.32 | 시간 5.81s | 오류 0 | 예산 소진 0 | 판정 {'RETRIEVE': 35, 'ABSTAIN': 26, 'ANSWER': 14}
v2 (r3): 정답 31/60 | 과잉 기권 0.18 | 호출 1.90 | 시간 4.83s | 오류 0 | 예산 소진 0 | 판정 {'RETRIEVE': 10, 'ABSTAIN': 13, 'ANSWER': 27}

v2만

In [20]:
diag_r3 = show_diagnosis("r3")
print()
print(diag_r3[(diag_r3["outcome"] != "정답") & diag_r3["gold_in_any"]].to_string(index=False))


[r3] 검색 경로 · 답 있는 문항 30개
gold_in_any  근거에 정답 있음  없음  All
outcome                        
기권                   6   3    9
오답                   9   2   11
정답                  10   0   10
All                 25   5   30

1차 근거에 이미 정답이 있던 24문항의 첫 판정: {'RETRIEVE': 4, 'ANSWER': 17, 'ABSTAIN': 3}

          qid             group outcome                 decisions  n_calls  gold_in_first  gold_in_any
hotpotqa-0040          hotpotqa      기권 RETRIEVE→RETRIEVE→ABSTAIN        4           True         True
hotpotqa-0044          hotpotqa      오답                    ANSWER        2           True         True
hotpotqa-0048          hotpotqa      기권 RETRIEVE→RETRIEVE→ABSTAIN        4           True         True
hotpotqa-0049          hotpotqa      오답                    ANSWER        2           True         True
hotpotqa-0051          hotpotqa      오답                    ANSWER        2           True         True
hotpotqa-0053          hotpotqa      기권          RETRIEVE→ABSTAIN        3           True 

## 16. 결과 내려받기 (r2 · r3)

두 라운드 로그를 레포의 `results/runs/dev/`에 올리고, 해석은 `docs/experiments/2026-11-26-a-m-dev-judge-prompt.md`에 채운다.


In [21]:
!cd results/runs/dev && zip -qr /content/a5b_dev_r2_r3.zip r2 r3
from google.colab import files
files.download("/content/a5b_dev_r2_r3.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 17. 로그를 노트북 출력으로 남기기 (`files.download` 대신)

VS Code에서 Colab 커널로 실행하면 `files.download()`가 띄우는 브라우저 다운로드(JavaScript)가 실행되지 않는다 (12·16의 출력이 `<IPython.core.display.Javascript object>`뿐인 이유).
대신 M 라운드 로그(`results/runs/dev/r*/`)를 tar.gz → base64로 출력한다. **실행 후 노트북을 저장(Ctrl+S)하면** 출력이 로컬 `.ipynb`에 남고, 거기서 꺼내 `sha256`을 확인한 뒤 레포의 `results/runs/dev/`에 푼다.


In [11]:
import base64, hashlib, io, tarfile
from pathlib import Path


def dump_logs():
    """M 로그가 있는 dev 라운드를 tar.gz -> base64로 출력한다. 노트북을 저장하면 로그가 파일에 남는다."""
    rounds = sorted(p.name for p in Path("results/runs/dev").iterdir() if p.is_dir() and (p / "m.jsonl").exists())
    buf = io.BytesIO()
    with tarfile.open(fileobj=buf, mode="w:gz") as tar:
        for rnd in rounds:
            tar.add(f"results/runs/dev/{rnd}", arcname=rnd)
    raw = buf.getvalue()
    data = base64.b64encode(raw).decode()
    print("rounds:", rounds, "| bytes:", len(raw), "| sha256:", hashlib.sha256(raw).hexdigest())
    print("A5B_LOGS_BEGIN")
    for i in range(0, len(data), 4000):
        print(data[i:i + 4000])
    print("A5B_LOGS_END")


dump_logs()


rounds: ['r2', 'r3', 'r4', 'r5'] | bytes: 38469 | sha256: 2d1b65053560b881a67618cf5e035d05d1eaccb07eebc3dfed87bfb97b325074
A5B_LOGS_BEGIN
...
A5B_LOGS_END


## 18. 판정 프롬프트 v3로 다시 실행 (r4)

15의 결과: v2는 과잉 기권 0.38 → 0.18, 호출 2.32 → 1.90으로 목표 지표가 좋아졌지만, SQuAD 답할 수 없음 정답이 7 → 4로 줄어 채택 기준을 통과하지 못했다.
새로 틀린 세 문항은 근거에 비슷한 문장이 있지만 질문과 한 부분이 어긋나는 SQuAD 2.0 함정 질문이다.

| qid | 질문 | 근거 | 어긋난 부분 |
|---|---|---|---|
| `squad2-0050` | phagocytosis의 역할, 영양분 획득에 쓰이기 **전** | 영양분 획득 수단으로 **진화했고** 이후 방어로 확장 | 순서 |
| `squad2-0053` | 소련이 군대로 **motivate**하려던 것 | 반란을 **suppress**하려 함 | 행동 |
| `squad2-0059` | plenary session의 **participants**인 비정부 단체 | **observers**로 참석이 허용될 수 있음 | 역할 |

v3는 v2에 두 가지만 더한다. ANSWER·RETRIEVE·빠진 정보 지시는 v2 그대로다.

| 자리 | v2 | v3 |
|---|---|---|
| ABSTAIN | the evidence does not contain the answer and another search would not help. | 같음 + **or it only says something similar that does not fit the question (for example a different action, role, order of events or condition)** |
| 서술 | 답이 되는 근거 부분을 인용하거나, 빠진 사실을 말함 | 인용한 뒤 **질문의 모든 세부와 맞는지 확인** |

**채택 기준 (실행 전에 정함, 15와 같은 기준):** 기준선은 v1(r2, 현재 M)이다. v3가 **유형별 정답 수에서 v1보다 줄지 않고**, 과잉 기권률과 호출 수(워밍업 제외)가 v1보다 좋아지면 채택한다.
같은 개발 세트 60문항으로 프롬프트를 세 번째 시험하는 것이라, 결과는 개발 세트에 맞춰진 쪽으로 기울 수 있다. 평가 세트 결과로만 효과를 주장한다.


In [23]:
assert Path("results/runs/dev/r3/m.jsonl").exists(), "r2·r3 로그가 있는 런타임에서 실행할 것"
!git pull --ff-only origin YongMin
!git log --oneline -1
!grep -n "fits every detail" src/meba/judge.py
!python -m pytest tests/test_judge.py tests/test_m_dev_smoke.py -q 2>&1 | tail -1
!python -m scripts.run_eval --bundle dev --round r4 --pipelines m


remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 13 (delta 12), reused 13 (delta 12), pack-reused 0 (from 0)
Unpacking objects: 100% (13/13), 10.73 KiB | 1.34 MiB/s, done.
From https://github.com/minyi-k03/Intelligence-Software-Project
 * branch            YongMin    -> FETCH_HEAD
   9d8f0a4..5924f88  YongMin    -> origin/YongMin
Updating 9d8f0a4..5924f88
Fast-forward
 .../experiments/2026-11-26-a-m-dev-judge-prompt.md |  75 +++-
 docs/progress.md                                   |   2 +-
 notebooks/verify_a5b.ipynb                         | 450 ++++++++++++++++++++-
 src/meba/judge.py                                  |  17 +-
 tests/test_judge.py                                |  10 +
 5 files changed, 518 insertions(+), 36 deletions(-)
5924f88 (HEAD -> YongMin, origin/YongMin) feat(meba): 근거 판정 지시문 v3 - 질문과 어긋나는 비슷한 문장은 ABSTAIN, r4 재실행 셀
51:    "it fits every detail of the question, or s

## 19. v1 · v2 · v3 비교와 채택 판정

13·15 셀(`load_round`, `round_summary`, `mcnemar_exact`, `show_diagnosis`)을 먼저 실행한 런타임에서 돌린다.


In [24]:
runs = {"v1 (r2)": load_round("r2"), "v2 (r3)": load_round("r3"), "v3 (r4)": load_round("r4")}
print(pd.concat({k: round_summary(d) for k, d in runs.items()}, axis=1).round(2).to_string())


def over_abstain(d):
    answerable = d["answer_kind"] != "unanswerable"
    return (d["abstained"] & answerable).sum() / answerable.sum()


def mean_calls(d):
    return d.loc[~d["is_warmup"], "n_calls"].mean()


for name, d in runs.items():
    c = d[~d["is_warmup"]]
    print(f"{name}: 정답 {int(d['correct'].sum())}/60 | 과잉 기권 {over_abstain(d):.2f} | 호출 {mean_calls(d):.2f}"
          f" | 시간 {c['latency_s'].mean():.2f}s | 오류 {int(d['error'].notna().sum())} | 예산 소진 {int(d['budget_exhausted'].sum())}"
          f" | 판정 {dict(Counter(x for xs in d['decisions'] for x in xs))}")

# 채택 판정 (기준선 v1)
base, cand = runs["v1 (r2)"], runs["v3 (r4)"]
per_type = round_summary(base)[["correct"]].join(round_summary(cand)[["correct"]], lsuffix="_v1", rsuffix="_v3")
dropped = per_type[per_type["correct_v3"] < per_type["correct_v1"]]
checks = {
    "유형별 정답 수가 v1보다 줄지 않음": dropped.empty,
    "과잉 기권률이 v1보다 낮음": over_abstain(cand) < over_abstain(base),
    "호출 수가 v1보다 적음": mean_calls(cand) < mean_calls(base),
}
print("\n[채택 판정: v3 vs 기준선 v1]")
print(per_type.to_string())
for k, ok in checks.items():
    print(f"  {'통과' if ok else '미달'}  {k}")
print("  => v3", "채택" if all(checks.values()) else "채택 기준 미달")


def show_flips(a_name, b_name):
    a, b = runs[a_name], runs[b_name]
    m = a.merge(b[["qid", "correct", "answer", "decisions", "route"]], on="qid", suffixes=("_a", "_b"))
    gained, lost = m[~m["correct_a"] & m["correct_b"]], m[m["correct_a"] & ~m["correct_b"]]
    print(f"\n=== {a_name} → {b_name}: 새로 맞힘 {len(gained)} / 새로 틀림 {len(lost)} | McNemar 정확 p = {mcnemar_exact(len(gained), len(lost)):.3f}")
    for title, part in (("새로 맞힘", gained), ("새로 틀림", lost)):
        for _, r in part.iterrows():
            print(f"  [{title}] {r['qid']:15s} {r['decisions_a']} {r['answer_a']!r} → {r['decisions_b']} {r['answer_b']!r} | 정답 {ITEMS[r['qid']]['answers']}")
    return m


m13 = show_flips("v1 (r2)", "v3 (r4)")
show_flips("v2 (r3)", "v3 (r4)")
math_rows = m13[m13["route_a"] == "math"]
print("\n수학 경로 결과가 v1과 같은가:", (math_rows["answer_a"] == math_rows["answer_b"]).all() and (math_rows["route_b"] == "math").all())

print("\n[SQuAD 답할 수 없음 10문항: 버전별 답]")
unans = runs["v1 (r2)"][runs["v1 (r2)"]["answer_kind"] == "unanswerable"]["qid"]
table = pd.DataFrame({name: d.set_index("qid").loc[unans, "answer"] for name, d in runs.items()})
print(table.to_string())

p1 = pd.DataFrame([json.loads(l) for l in Path("results/runs/dev/r1/p1.jsonl").open(encoding="utf-8")])
p1c, v3c = p1[~p1["is_warmup"]], cand[~cand["is_warmup"]]
print(f"\nP1(r1): 정답 {int(p1['correct'].sum())}/60 | 호출 {p1c['n_calls'].mean():.2f} | 시간 평균 {p1c['latency_s'].mean():.2f}s"
      f" · 중앙값 {p1c['latency_s'].median():.2f}s | 시간 초과 {int(p1['timeout'].sum())}")
print(f"M v3(r4): 정답 {int(cand['correct'].sum())}/60 | 호출 {v3c['n_calls'].mean():.2f} | 시간 평균 {v3c['latency_s'].mean():.2f}s"
      f" · 중앙값 {v3c['latency_s'].median():.2f}s | 시간 초과 {int(cand['timeout'].sum())}")

print()
diag_r4 = show_diagnosis("r4")
print()
print(diag_r4[(diag_r4["outcome"] != "정답") & diag_r4["gold_in_any"]].to_string(index=False))


                    v1 (r2)                                     v2 (r3)                                     v3 (r4)                                    
                          n correct abstained n_calls latency_s       n correct abstained n_calls latency_s       n correct abstained n_calls latency_s
group                                                                                                                                                  
gsm8k                    20      17         0    1.30      7.84      20      17         0    1.30      7.78      20      17         0     1.3      7.73
hotpotqa                 20       3        13    2.89      5.08      20       5         8    2.26      3.70      20       3        10     2.0      3.11
squad2-answerable        10       2         6    3.10      5.40      10       5         1    2.30      3.37      10       5         3     2.1      3.00
squad2-unanswerable      10       7         7    2.50      3.57      10       4         

## 20. 로그 남기기 (r2 · r3 · r4)

17과 같다. 실행한 뒤 **노트북을 저장(Ctrl+S)** 한다.


In [25]:
dump_logs()


rounds: ['r2', 'r3', 'r4'] | bytes: 25301 | sha256: 39d3caa987c5e285ceedae7a4759398b9e388dbcd82b553f3d62f011a73c03bd
A5B_LOGS_BEGIN
...
A5B_LOGS_END


## 21. 추가 실험 (M 구성 = 판정 지시문 v2로 동결)

v1·v2·v3 비교 뒤 팀 결정으로 v2로 동결했다. 가설(P1보다 정확도는 높고 호출 수·시간은 P1 이하)을 공정하게 판단하려고 네 가지를 더 본다.

| 절 | 실험 | 왜 |
|---|---|---|
| 22 | **같은 세션·같은 설정으로 P1과 M(v2)을 다시 실행 (`r5`)** | 지금 비교 대상인 P1(`r1`)은 다른 세션이고 설정 스냅샷도 다르다(`judge` 섹션 없음). 시간 초과 4건 때문에 호출 수가 낮게 잡혀 있다. M을 함께 돌리면 `r3`와 답이 같은지로 재현성도 확인된다 |
| 23 | 실패 단계 분해 (M·P1) | 어느 단계(검색·판정·답 형식)에서 틀리는지 나눠, 하네스가 무엇을 줄였는지 보인다 |
| 24 | 문항 부트스트랩 95% 신뢰구간 | 보고 형식 "정확도 +Δ, 호출 ×k, 시간 ×t"에 불확실성을 붙인다. McNemar는 정확도의 p값만 준다 |
| 24 | 시간은 평균과 중앙값 둘 다 | P1 평균은 120초 시간 초과가 끌어올린다 |

- 23·24는 모델을 부르지 않는 로그 분석이다. 13·15·17 셀(`ITEMS`, `CORPUS`, `render`, `mcnemar_exact`, `dump_logs`)을 먼저 실행한 런타임에서 돌린다.
- `r5`는 `p1,m` 두 파이프라인이라 순서는 P1 → M이고, 파이프라인마다 첫 문항이 워밍업이다.


In [26]:
!git pull --ff-only origin YongMin
!git log --oneline -1
!grep -n "M 구성 동결(11/27)은 v2" src/meba/judge.py
!python -m pytest tests/test_judge.py tests/test_m_dev_smoke.py -q 2>&1 | tail -1
!python -m scripts.run_eval --bundle dev --round r5 --pipelines p1,m


remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 33 (delta 22), reused 33 (delta 22), pack-reused 0 (from 0)
Unpacking objects: 100% (33/33), 92.97 KiB | 3.44 MiB/s, done.
From https://github.com/minyi-k03/Intelligence-Software-Project
 * branch            YongMin    -> FETCH_HEAD
   5924f88..734bcbf  YongMin    -> origin/YongMin
Updating 5924f88..734bcbf
error: The following untracked working tree files would be overwritten by merge:
	results/runs/dev/r2/RUN_META.json
	results/runs/dev/r2/m.jsonl
	results/runs/dev/r3/RUN_META.json
	results/runs/dev/r3/m.jsonl
	results/runs/dev/r4/RUN_META.json
	results/runs/dev/r4/m.jsonl
Please move or remove them before you merge.
Aborting
5924f88 (HEAD -> YongMin) feat(meba): 근거 판정 지시문 v3 - 질문과 어긋나는 비슷한 문장은 ABSTAIN, r4 재실행 셀
48 passed in 1.99s
[p1 r5]   1 hotpotqa-0040 ok 2.8s
[p1 r5]   2 hotpotqa-0041 ok 2.1s
[p1 r5]   3 hotpotqa-0042 wrong 1.8s
[p1 

## 22. 재현성과 P1 재측정 확인

- **재현성:** M(v2)은 `r3`와 같은 지시문·설정이다. temperature 0이면 문항별 답이 같아야 한다. 다르면 본 측정 3라운드의 정확도도 라운드마다 흔들릴 수 있다는 뜻이다.
- **P1:** `r1`(이전 세션)과 `r5`(이번 세션)의 정답·호출·시간 초과를 비교한다. 시간 초과 문항은 호출 기록을 수거하지 못해 `n_calls=0`으로 남는다 (EVALUATION §4).


In [12]:
def load_log(rnd, pipeline):
    """라운드·파이프라인의 로그를 DataFrame으로 읽는다."""
    rows = [json.loads(l) for l in Path(f"results/runs/dev/{rnd}/{pipeline}.jsonl").open(encoding="utf-8")]
    df = pd.DataFrame(rows)
    df["group"] = df.apply(lambda r: f"{r['type']}-{r['subtype']}" if r["type"] == "squad2" else r["type"], axis=1)
    return df


m_old, m_new = load_log("r3", "m"), load_log("r5", "m")
same = m_old.merge(m_new[["qid", "answer", "correct"]], on="qid", suffixes=("_r3", "_r5"))
diff = same[same["answer_r3"] != same["answer_r5"]]
print(f"[재현성] M v2 r3 vs r5: 답이 같은 문항 {len(same) - len(diff)}/{len(same)}"
      f" | 정답 {int(same['correct_r3'].sum())} vs {int(same['correct_r5'].sum())}")
for _, r in diff.iterrows():
    print(f"  {r['qid']:15s} {r['answer_r3']!r} -> {r['answer_r5']!r}")

print()
for rnd in ("r1", "r5"):
    p = load_log(rnd, "p1")
    c = p[~p["is_warmup"]]
    print(f"[P1 {rnd}] 정답 {int(p['correct'].sum())}/60 | 호출 {c['n_calls'].mean():.2f} | 시간 평균 {c['latency_s'].mean():.2f}s"
          f" · 중앙값 {c['latency_s'].median():.2f}s | 시간 초과 {int(p['timeout'].sum())} {list(p.loc[p['timeout'], 'qid'])}"
          f" | 오류 {int((p['error'].notna() & ~p['timeout']).sum())}")
p5 = load_log("r5", "p1")
for _, r in p5[p5["error"].notna()].iterrows():
    print(f"  {r['qid']:15s} timeout={r['timeout']} error={str(r['error'])[:100]}")


[재현성] M v2 r3 vs r5: 답이 같은 문항 51/60 | 정답 31 vs 29
  hotpotqa-0046   'Montana State University' -> 'UNANSWERABLE'
  hotpotqa-0050   'board game' -> 'UNANSWERABLE'
  hotpotqa-0056   'large chains' -> 'franchise-owned chain'
  squad2-0040     'final years of the Yuan dynasty' -> 'UNANSWERABLE'
  squad2-0041     'garrisons' -> 'UNANSWERABLE'
  squad2-0043     'every year between 1346 and 1671' -> 'the plague was present somewhere in Europe in every year between 1346 and 1671'
  squad2-0053     'to suppress an Islamic rebellion against an allied Marxist regime' -> 'to fight against the Islamic forces in Afghanistan'
  squad2-0056     'tears and urine' -> 'mucus'
  squad2-0057     'a drive shaft' -> 'drive shaft'

[P1 r1] 정답 21/60 | 호출 1.75 | 시간 평균 9.02s · 중앙값 2.77s | 시간 초과 4 ['hotpotqa-0040', 'hotpotqa-0049', 'squad2-0045', 'squad2-0051'] | 오류 2
[P1 r5] 정답 22/60 | 호출 1.78 | 시간 평균 10.58s · 중앙값 2.33s | 시간 초과 4 ['hotpotqa-0044', 'hotpotqa-0049', 'squad2-0045', 'squad2-0051'] | 오류 0
  hotpotqa-

## 23. 실패 단계 분해 (답이 있는 HotpotQA·SQuAD 30문항)

문항 유형은 분석에만 쓴다 (모델에는 넘기지 않는다). 근거는 로그로 복원한다: M은 압축 근거(`kept_windows`), P1은 검색한 문단 전체.

| 단계 | 뜻 | 책임 |
|---|---|---|
| 정답 | | |
| 검색 안 함 | 검색 없이 답함 (P1 직답, M 수학 경로로 잘못 감) | 라우팅·도구 선택 |
| 근거에 정답 없음 | 검색·압축이 정답 문장을 못 가져옴 | 검색·압축 |
| 근거 있는데 기권 | | 판정 |
| 답 범위 어긋남 | 오답이지만 F1 ≥ 0.5 (예: `Pac-12 Conference` vs `Pac-12`) | 답 형식 (모든 파이프라인 공통) |
| 근거 있는데 오답 | F1 < 0.5 | 판정·답 추출 |


In [13]:
def evidence_text(row):
    """판정·답에 쓰인 근거를 복원한다. 압축 기록이 있으면 압축 근거, 없으면 검색 문단 전체."""
    parts = []
    windows = row.get("kept_windows") or []
    for i, doc_ids in enumerate(row.get("retrieved_doc_ids") or []):
        paras = [CORPUS[d] for d in doc_ids if d in CORPUS]
        kept = windows[i] if i < len(windows) else None
        if kept:
            by_id = {w.window_id: w for w in build_windows(paras, WINDOW)}
            parts.append(render(paras, [by_id[k] for k in kept if k in by_id]))
        else:
            parts.append("\n".join(f"{p['title']}: {' '.join(p['sentences'])}" for p in paras))
    return _normalize(" ".join(parts))


STAGES = ["정답", "검색 안 함", "근거에 정답 없음", "근거 있는데 기권", "답 범위 어긋남", "근거 있는데 오답"]


def failure_stages(df):
    """답이 있는 HotpotQA·SQuAD 문항마다 실패 단계를 붙인다."""
    sub = df[(df["type"] != "gsm8k") & (df["answer_kind"] != "unanswerable")].copy()

    def stage(r):
        if r["correct"]:
            return "정답"
        if not r.get("retrieved_doc_ids"):
            return "검색 안 함"
        ev = evidence_text(r)
        if not any(_normalize(g) in ev for g in ITEMS[r["qid"]]["answers"]):
            return "근거에 정답 없음"
        if r["abstained"]:
            return "근거 있는데 기권"
        return "답 범위 어긋남" if (r["f1"] or 0) >= 0.5 else "근거 있는데 오답"

    sub["stage"] = sub.apply(stage, axis=1)
    return sub


logs = {"M v1 (r2)": load_log("r2", "m"), "M v2 (r3)": load_log("r3", "m"), "M v2 (r5)": load_log("r5", "m"),
        "P1 (r1)": load_log("r1", "p1"), "P1 (r5)": load_log("r5", "p1")}
staged = {name: failure_stages(d) for name, d in logs.items()}
table = pd.DataFrame({name: s["stage"].value_counts() for name, s in staged.items()}).reindex(STAGES).fillna(0).astype(int)
print(table.to_string())
print()
for name in ("M v2 (r5)", "P1 (r5)"):
    s = staged[name]
    near = s[s["stage"] == "답 범위 어긋남"]
    print(f"{name} 답 범위 어긋남:", [(r.qid, r.answer, ITEMS[r.qid]["answers"][0], round(r.f1, 2)) for r in near.itertuples()])


           M v1 (r2)  M v2 (r3)  M v2 (r5)  P1 (r1)  P1 (r5)
stage                                                       
정답                 5         10          8        8        8
검색 안 함             0          0          0        4        5
근거에 정답 없음          6          5          6        4        4
근거 있는데 기권         14          6          8        7        5
답 범위 어긋남           4          5          5        4        5
근거 있는데 오답          1          4          3        3        3

M v2 (r5) 답 범위 어긋남: [('hotpotqa-0049', 'Unbreakable Kimmy Schmidt', 'Kimberly Couger "Kimmy" Schmidt', 0.57), ('hotpotqa-0056', 'franchise-owned chain', 'restaurant chain', 0.5), ('hotpotqa-0059', 'Bhabananda Deka', 'Prof Bhabananda Deka', 0.8), ('squad2-0046', 'French Louisiana west of the Mississippi River', 'French Louisiana west of the Mississippi River (including New Orleans) to its ally Spain', 0.91), ('squad2-0049', 'Pac-12 Conference', 'Pac-12', 0.67)]
P1 (r5) 답 범위 어긋남: [('hotpotqa-0056', 'chain re

## 24. M(v2) vs P1: 문항 부트스트랩 95% 신뢰구간 (`r5`, 같은 세션)

- 문항을 복원 추출(paired, 10,000회)해 M과 P1을 같은 문항 묶음으로 비교한다. 시드는 설정의 `data.seed`.
- 정확도 차이 Δ는 60문항 전체, 호출 비율 k와 시간 비율 t는 양쪽 모두 워밍업이 아닌 문항 (EVALUATION §4).
- k = M 호출 합 / P1 호출 합. **가설 "호출 수 P1 이하"는 k의 95% 구간 상한이 1 이하일 때 지지된다.** 구간이 1을 걸치면 판단 불가다.
- 시간은 평균 비율과 중앙값 비율을 함께 낸다. P1 시간 초과 문항은 호출이 0으로 기록되므로, 시간 초과 문항을 양쪽에서 뺀 민감도 분석도 함께 낸다.


In [14]:
import numpy as np

SEED = get_settings().data.seed
B = 10_000


def paired_bootstrap(m, p, label):
    both = m.merge(p, on="qid", suffixes=("_m", "_p"))
    cost = both[~both["is_warmup_m"] & ~both["is_warmup_p"]]
    rng = np.random.default_rng(SEED)
    acc_m, acc_p = both["correct_m"].to_numpy(float), both["correct_p"].to_numpy(float)
    cm, cp = cost["n_calls_m"].to_numpy(float), cost["n_calls_p"].to_numpy(float)
    tm, tp = cost["latency_s_m"].to_numpy(float), cost["latency_s_p"].to_numpy(float)

    def ci(fn, cols):
        n = len(cols[0])
        idx = rng.integers(0, n, size=(B, n))
        vals = np.array([fn(*(c[i] for c in cols)) for i in idx])
        lo, hi = np.percentile(vals, [2.5, 97.5])
        return fn(*cols), lo, hi

    stats = {
        "정확도 차이 Δ (M − P1)": ci(lambda a, b: a.mean() - b.mean(), (acc_m, acc_p)),
        "호출 비율 k (M / P1)": ci(lambda a, b: a.sum() / b.sum(), (cm, cp)),
        "시간 비율 t, 평균 (M / P1)": ci(lambda a, b: a.sum() / b.sum(), (tm, tp)),
        "시간 비율 t, 중앙값 (M / P1)": ci(lambda a, b: np.median(a) / np.median(b), (tm, tp)),
    }
    b_only = int((both["correct_m"] & ~both["correct_p"]).sum())
    c_only = int((~both["correct_m"] & both["correct_p"]).sum())
    print(f"[{label}] 문항 {len(both)} (비용 {len(cost)}) | M만 정답 {b_only} / P1만 정답 {c_only}"
          f" | McNemar 정확 p = {mcnemar_exact(b_only, c_only):.3f}")
    for name, (point, lo, hi) in stats.items():
        print(f"  {name:24s} {point:+.3f}  [95% {lo:+.3f}, {hi:+.3f}]" if "Δ" in name
              else f"  {name:24s} ×{point:.2f}  [95% ×{lo:.2f}, ×{hi:.2f}]")
    k_hi = stats["호출 비율 k (M / P1)"][2]
    print("  => 호출 수 P1 이하:", "지지 (k 상한 ≤ 1)" if k_hi <= 1 else
          ("반대 (k 하한 > 1)" if stats["호출 비율 k (M / P1)"][1] > 1 else "판단 불가 (구간이 1을 걸침)"))


m5, p5 = load_log("r5", "m"), load_log("r5", "p1")
paired_bootstrap(m5, p5, "본 비교: M v2 r5 vs P1 r5")
timed_out = set(m5.loc[m5["timeout"], "qid"]) | set(p5.loc[p5["timeout"], "qid"])
if timed_out:
    print()
    paired_bootstrap(m5[~m5["qid"].isin(timed_out)], p5[~p5["qid"].isin(timed_out)],
                     f"민감도: 시간 초과 {len(timed_out)}문항을 양쪽에서 뺌")
print()
print("유형별 정답 (M v2 r5 / P1 r5):")
print(pd.concat({"M": m5.groupby("group")["correct"].sum(), "P1": p5.groupby("group")["correct"].sum()}, axis=1).to_string())


[본 비교: M v2 r5 vs P1 r5] 문항 60 (비용 59) | M만 정답 13 / P1만 정답 6 | McNemar 정확 p = 0.167
  정확도 차이 Δ (M − P1)        +0.117  [95% -0.017, +0.250]
  호출 비율 k (M / P1)         ×1.00  [95% ×0.90, ×1.12]
  시간 비율 t, 평균 (M / P1)     ×0.44  [95% ×0.23, ×1.14]
  시간 비율 t, 중앙값 (M / P1)    ×1.41  [95% ×0.94, ×1.81]
  => 호출 수 P1 이하: 판단 불가 (구간이 1을 걸침)

[민감도: 시간 초과 4문항을 양쪽에서 뺌] 문항 56 (비용 55) | M만 정답 11 / P1만 정답 6 | McNemar 정확 p = 0.332
  정확도 차이 Δ (M − P1)        +0.089  [95% -0.054, +0.232]
  호출 비율 k (M / P1)         ×0.92  [95% ×0.85, ×1.01]
  시간 비율 t, 평균 (M / P1)     ×1.82  [95% ×1.43, ×2.36]
  시간 비율 t, 중앙값 (M / P1)    ×1.46  [95% ×1.00, ×2.04]
  => 호출 수 P1 이하: 판단 불가 (구간이 1을 걸침)

유형별 정답 (M v2 r5 / P1 r5):
                      M  P1
group                      
gsm8k                17  10
hotpotqa              3   5
squad2-answerable     5   3
squad2-unanswerable   4   4


## 25. 로그 남기기 (r2 ~ r5)

17과 같다. 실행한 뒤 **노트북을 저장(Ctrl+S)** 한다. `r5`에는 `p1.jsonl`도 함께 담긴다.


In [30]:
dump_logs()


rounds: ['r2', 'r3', 'r4', 'r5'] | bytes: 38537 | sha256: 647dd93cb403f98665c19e8a1a186ee178d3405507baf9e3626ae83db6e622f4
A5B_LOGS_BEGIN
...
A5B_LOGS_END


## 26. r5의 M은 v3로 돌았다 → 동결 M(v2)과 P1을 `r6`에서 다시 실행

**무슨 일이 있었나.** §22의 `git pull`이 원격 커밋(`734bcbf`, v2 복원)을 받아 오기만 하고 적용에 실패했다. 레포에 새로 커밋된 `results/runs/dev/r2~r4`와 이름이 같은 추적 안 된 파일이 런타임에 있어서, git이 덮어쓰기를 거부했다. 그래서 HEAD가 `5924f88`(v3)에 남았다.
- r5 M의 답은 r4(v3)와 **60/60 같다**. `RUN_META`의 커밋도 `5924f88`이다. **r5의 M은 v3 측정이고, 동결 M의 측정이 아니다.**
- r5의 **P1은 유효하다.** 두 커밋의 코드 차이는 `src/meba/judge.py`뿐이고, P1은 판정 단계를 쓰지 않는다.
- 덤으로 **결정성이 확인됐다.** 같은 코드(v3)를 다른 실행(r4, r5)에서 돌려 답 60개가 같았다.

**새 런타임에서 실행할 셀 (Run All 금지).** 8(`r2`)·14·18·22의 실행 셀은 다시 돌리지 않는다.
1. §0~6: GPU 확인 → 코드 받기 → 의존성 → Ollama → 모델 → GPU 로드 확인 → 테스트 → 해시
2. §13(`a5b-30`) · §15(`a5b-34`) · §17(`a5b-39`): 분석 함수 정의. r2~r5 로그는 git에서 받는다
3. §22~24의 분석 셀(`a5b-49`, `a5b-51`, `a5b-53`): 함수 정의용. 출력의 r5 M은 v3라는 점에 주의
4. **아래 `a5b-57`**: pull이 실제로 적용됐는지 확인한 뒤 `r6`에 P1과 M을 같은 세션으로 실행한다 (약 15분). `r6`은 4 % 2 = 1칸 밀려 **M → P1** 순서다. r5(P1 → M)와 순서가 반대다
5. §27 · §28


In [15]:
import shutil
import subprocess
from pathlib import Path


def sh(*args):
    return subprocess.run(list(args), capture_output=True, text=True).stdout.strip()


# 예전 런타임에서 만든, 추적 안 된 라운드 폴더가 pull을 막지 않게 옮긴다 (같은 로그가 git에 있다).
sh("git", "fetch", "origin")
untracked = {Path(l[3:]).parts[3] for l in sh("git", "status", "--porcelain", "--untracked-files=all", "results/runs/dev").splitlines()
             if l.startswith("?? ") and len(Path(l[3:]).parts) > 4}
for rnd in sorted(untracked):
    if sh("git", "ls-tree", "-d", "origin/YongMin", f"results/runs/dev/{rnd}"):
        Path("/content/stale_runs").mkdir(exist_ok=True)
        shutil.move(f"results/runs/dev/{rnd}", f"/content/stale_runs/{rnd}")
        print("옮김:", rnd, "-> /content/stale_runs/")

print(sh("git", "pull", "--ff-only", "origin", "YongMin"))
head, origin = sh("git", "rev-parse", "HEAD"), sh("git", "rev-parse", "origin/YongMin")
assert head == origin, f"pull이 적용되지 않았다: HEAD {head[:7]} != origin/YongMin {origin[:7]} -> git status를 확인할 것"
assert "M 구성 동결(11/27)은 v2" in Path("src/meba/judge.py").read_text(encoding="utf-8"), "judge.py가 동결 v2가 아니다"
print("OK: HEAD", head[:7], "= origin/YongMin, judge.py = v2")
!python -m pytest tests/test_judge.py tests/test_m_dev_smoke.py -q 2>&1 | tail -1
!python -m scripts.run_eval --bundle dev --round r6 --pipelines p1,m


Already up to date.
OK: HEAD 17e71c7 = origin/YongMin, judge.py = v2
47 passed in 2.04s
[m r6]   1 hotpotqa-0040 wrong 17.3s
[m r6]   2 hotpotqa-0041 ok 3.6s
[m r6]   3 hotpotqa-0042 wrong 2.7s
[m r6]   4 hotpotqa-0043 wrong 3.7s
[m r6]   5 hotpotqa-0044 wrong 2.5s
[m r6]   6 hotpotqa-0045 wrong 5.4s
[m r6]   7 hotpotqa-0046 ok 3.2s
[m r6]   8 hotpotqa-0047 wrong 3.7s
[m r6]   9 hotpotqa-0048 wrong 16.1s
[m r6]  10 hotpotqa-0049 wrong 3.3s
[m r6]  11 hotpotqa-0050 ok 10.2s
[m r6]  12 hotpotqa-0051 wrong 2.2s
[m r6]  13 hotpotqa-0052 ok 3.5s
[m r6]  14 hotpotqa-0053 wrong 7.7s
[m r6]  15 hotpotqa-0054 wrong 3.3s
[m r6]  16 hotpotqa-0055 wrong 2.9s
[m r6]  17 hotpotqa-0056 wrong 2.8s
[m r6]  18 hotpotqa-0057 ok 3.9s
[m r6]  19 hotpotqa-0058 wrong 2.5s
[m r6]  20 hotpotqa-0059 wrong 3.3s
[m r6]  21 squad2-0040 wrong 12.4s
[m r6]  22 squad2-0041 ok 2.4s
[m r6]  23 squad2-0042 wrong 2.7s
[m r6]  24 squad2-0043 wrong 3.5s
[m r6]  25 squad2-0044 ok 6.4s
[m r6]  26 squad2-0045 ok 2.9s
[m r6]  

## 27. 동결 M(v2) vs P1: 같은 세션 `r6`

- 재현성: r6 M vs r3 M (둘 다 v2). P1은 r1·r5·r6 세 번의 시간 초과 문항을 비교한다 (P1의 멈춤이 문항에 따라 반복되는가).
- 실패 단계: M r6, P1 r6
- 부트스트랩 95% 구간: **M r6 vs P1 r6** (같은 세션). 이 결과를 실험 기록과 보고서의 dev 수치로 쓴다.


In [16]:
m6, p6 = load_log("r6", "m"), load_log("r6", "p1")
assert json.load(open("results/runs/dev/r6/RUN_META.json"))["git_commit"] == sh("git", "rev-parse", "HEAD"), "r6가 지금 커밋으로 돌지 않았다"

same = load_log("r3", "m").merge(m6[["qid", "answer"]], on="qid", suffixes=("_r3", "_r6"))
diff = same[same["answer_r3"] != same["answer_r6"]]
print(f"[재현성] 동결 M v2 r3 vs r6: 답이 같은 문항 {len(same) - len(diff)}/{len(same)}")
for _, r in diff.iterrows():
    print(f"  {r['qid']:15s} {r['answer_r3']!r} -> {r['answer_r6']!r}")

print()
for rnd in ("r1", "r5", "r6"):
    p = load_log(rnd, "p1")
    c = p[~p["is_warmup"]]
    print(f"[P1 {rnd}] 정답 {int(p['correct'].sum())}/60 | 호출 {c['n_calls'].mean():.2f} | 시간 평균 {c['latency_s'].mean():.2f}s"
          f" · 중앙값 {c['latency_s'].median():.2f}s | 시간 초과 {sorted(p.loc[p['timeout'], 'qid'])}")

print()
table = pd.DataFrame({name: failure_stages(d)["stage"].value_counts() for name, d in
                      (("M v2 (r3)", load_log("r3", "m")), ("M v2 (r6)", m6), ("P1 (r5)", load_log("r5", "p1")), ("P1 (r6)", p6))})
print(table.reindex(STAGES).fillna(0).astype(int).to_string())

print()
paired_bootstrap(m6, p6, "본 비교: 동결 M v2 r6 vs P1 r6 (같은 세션)")
timed_out = set(m6.loc[m6["timeout"], "qid"]) | set(p6.loc[p6["timeout"], "qid"])
if timed_out:
    print()
    paired_bootstrap(m6[~m6["qid"].isin(timed_out)], p6[~p6["qid"].isin(timed_out)],
                     f"민감도: 시간 초과 {len(timed_out)}문항을 양쪽에서 뺌")
print()
print(pd.concat({"M": m6.groupby("group")["correct"].sum(), "P1": p6.groupby("group")["correct"].sum()}, axis=1).to_string())


[재현성] 동결 M v2 r3 vs r6: 답이 같은 문항 60/60

[P1 r1] 정답 21/60 | 호출 1.75 | 시간 평균 9.02s · 중앙값 2.77s | 시간 초과 ['hotpotqa-0040', 'hotpotqa-0049', 'squad2-0045', 'squad2-0051']
[P1 r5] 정답 22/60 | 호출 1.78 | 시간 평균 10.58s · 중앙값 2.33s | 시간 초과 ['hotpotqa-0044', 'hotpotqa-0049', 'squad2-0045', 'squad2-0051']
[P1 r6] 정답 22/60 | 호출 1.78 | 시간 평균 11.09s · 중앙값 2.76s | 시간 초과 ['hotpotqa-0044', 'hotpotqa-0049', 'squad2-0045', 'squad2-0051']

           M v2 (r3)  M v2 (r6)  P1 (r5)  P1 (r6)
stage                                            
정답                10         10        8        8
검색 안 함             0          0        5        5
근거에 정답 없음          5          5        4        4
근거 있는데 기권          6          6        5        5
답 범위 어긋남           5          5        5        5
근거 있는데 오답          4          4        3        3

[본 비교: 동결 M v2 r6 vs P1 r6 (같은 세션)] 문항 60 (비용 59) | M만 정답 13 / P1만 정답 4 | McNemar 정확 p = 0.049
  정확도 차이 Δ (M − P1)        +0.150  [95% +0.017, +0.283]
  호출 비율 k (M / P1)         

## 28. 로그 남기기 (r2 ~ r6)

실행한 뒤 **노트북을 저장(Ctrl+S)** 한다.


In [17]:
dump_logs()


rounds: ['r2', 'r3', 'r4', 'r5', 'r6'] | bytes: 52299 | sha256: e1627ce1017229b7660d6f404d9d18345a5ab37cb769d51a032f52fca3d025ab
A5B_LOGS_BEGIN
...
A5B_LOGS_END


## 29. 절제 실험 `r7`: 이득이 어느 부품에서 오는가

**질문.** r6에서 M(v2)이 P1보다 정확했다(31 vs 22). 그런데 이득의 대부분이 GSM8K(17 vs 10)에서 나왔다. 이 이득은 라우터의 수학 경로 때문인가, 검색 경로의 부품(재검색·판정·압축) 때문인가?

**방법.** 동결 M(v2)과 부품을 하나씩 끈 변형 4개를 **같은 라운드·같은 세션**에서 돌린다(dev 60문항). 변형은 `src/meba/graph.py`에 이미 있다. 코드와 지시문은 바꾸지 않는다. 이 실험은 M을 고치는 튜닝이 아니라 원인을 설명하는 분석이다.

| 변형 | 바뀌는 것 | 실행 전 예상 |
|---|---|---|
| `m_minus_router` | 라우터 호출 없이 **모든 질문을 검색 경로로** (GSM8K도 검색) | GSM8K 정답이 크게 줄고(5개 이상), 호출 수는 줄어든다 |
| `m_minus_requery` | RETRIEVE를 없애고 **검색 한 번**만 | 호출 수가 줄고 정답은 ±2 안에서 변한다 (HotpotQA가 먼저 나빠질 수 있다) |
| `m_minus_judge` | 판정에서 **ABSTAIN만 제거** (재검색은 유지) | 과잉 기권은 0이 되고 SQuAD 답할 수 없음 정답이 4에서 크게 줄어든다. 답 있는 문항 정답은 늘 수 있다 |
| `m_minus_compress` | 압축 없이 **상위 k개 문단을 통째로** | 입력 토큰과 시간이 늘고 정확도는 비슷하다 |

**읽는 규칙 (실행 전에 정함).**
- 60문항이라 정답 수 ±2는 노이즈다. 부품이 기여한다고 말하려면 **M − 변형의 정확도 차이 95% 구간이 0을 넘어야** 한다. 구간이 0을 걸치면 "기여를 주장할 수 없음"으로 적는다.
- 위 예상이 틀려도 그대로 기록한다. 예상과 다른 결과도 결과다.
- 이 결과로 M의 구성을 바꾸지 않는다(동결). 공식 절제 측정(평가 세트)의 예행연습이자 보고서의 설명 근거다.
- `m_minus_cap`, `m_minus_loop_breaker`는 정확도 이득의 원인과 거리가 멀어 이번에는 뺐다. 공식 절제 측정에서 돌린다.

**새 런타임에서 실행할 셀 (Run All 금지).** 8(`r2`)·14·18·22·26(`r6`) 실행 셀은 다시 돌리지 않는다.
1. §0~6: GPU 확인 → 코드 받기 → 의존성 → Ollama → 모델 → GPU 로드 확인 → 테스트 → 해시
2. §13(`a5b-30`) · §15(`a5b-34`) · §17(`a5b-39`) · §22(`a5b-49`): 분석 함수 정의
3. **아래 `a5b-63`**: pull 확인 뒤 `r7` 실행 (5개 파이프라인, 약 50~60분. `m_minus_compress`는 입력이 길어 더 느리다)
4. §30 · §31

이미 같은 런타임에서 r6까지 돌렸다면 1·2는 건너뛰고 `a5b-63`부터 돌린다.


In [18]:
import shutil
import subprocess
from pathlib import Path


def sh(*args):
    return subprocess.run(list(args), capture_output=True, text=True).stdout.strip()


# 예전 런타임에서 만든, 추적 안 된 라운드 폴더가 pull을 막지 않게 옮긴다 (같은 로그가 git에 있다).
sh("git", "fetch", "origin")
untracked = {Path(l[3:]).parts[3] for l in sh("git", "status", "--porcelain", "--untracked-files=all", "results/runs/dev").splitlines()
             if l.startswith("?? ") and len(Path(l[3:]).parts) > 4}
for rnd in sorted(untracked):
    if sh("git", "ls-tree", "-d", "origin/YongMin", f"results/runs/dev/{rnd}"):
        Path("/content/stale_runs").mkdir(exist_ok=True)
        shutil.move(f"results/runs/dev/{rnd}", f"/content/stale_runs/{rnd}")
        print("옮김:", rnd, "-> /content/stale_runs/")

print(sh("git", "pull", "--ff-only", "origin", "YongMin"))
head, origin = sh("git", "rev-parse", "HEAD"), sh("git", "rev-parse", "origin/YongMin")
assert head == origin, f"pull이 적용되지 않았다: HEAD {head[:7]} != origin/YongMin {origin[:7]} -> git status를 확인할 것"
assert "M 구성 동결(11/27)은 v2" in Path("src/meba/judge.py").read_text(encoding="utf-8"), "judge.py가 동결 v2가 아니다"
assert "m_minus_compress" in Path("scripts/run_eval.py").read_text(encoding="utf-8"), "절제 변형이 등록되어 있지 않다"
print("OK: HEAD", head[:7], "= origin/YongMin, judge.py = v2, 절제 변형 등록됨")
!python -m pytest tests/test_graph.py tests/test_judge.py tests/test_m_dev_smoke.py -q 2>&1 | tail -1
!python -m scripts.run_eval --bundle dev --round r7 --pipelines m,m_minus_router,m_minus_requery,m_minus_judge,m_minus_compress


Already up to date.
OK: HEAD 17e71c7 = origin/YongMin, judge.py = v2, 절제 변형 등록됨
91 passed in 3.01s
[m_minus_router r7]   1 hotpotqa-0040 wrong 12.3s
[m_minus_router r7]   2 hotpotqa-0041 ok 4.3s
[m_minus_router r7]   3 hotpotqa-0042 wrong 3.1s
[m_minus_router r7]   4 hotpotqa-0043 wrong 4.1s
[m_minus_router r7]   5 hotpotqa-0044 wrong 2.6s
[m_minus_router r7]   6 hotpotqa-0045 wrong 5.4s
[m_minus_router r7]   7 hotpotqa-0046 ok 3.0s
[m_minus_router r7]   8 hotpotqa-0047 wrong 3.5s
[m_minus_router r7]   9 hotpotqa-0048 wrong 12.0s
[m_minus_router r7]  10 hotpotqa-0049 wrong 2.6s
[m_minus_router r7]  11 hotpotqa-0050 ok 7.8s
[m_minus_router r7]  12 hotpotqa-0051 wrong 1.9s
[m_minus_router r7]  13 hotpotqa-0052 ok 3.1s
[m_minus_router r7]  14 hotpotqa-0053 wrong 7.2s
[m_minus_router r7]  15 hotpotqa-0054 wrong 3.1s
[m_minus_router r7]  16 hotpotqa-0055 wrong 2.9s
[m_minus_router r7]  17 hotpotqa-0056 wrong 2.6s
[m_minus_router r7]  18 hotpotqa-0057 ok 3.7s
[m_minus_router r7]  19 hotpotqa

## 30. 절제 결과: 각 부품을 빼면 무엇이 바뀌나

- 재현성: `r7`의 M은 `r3`·`r6`의 M(v2)과 답이 같아야 한다.
- 요약표: 변형별 정답(유형별), 과잉 기권, 호출, 입력 토큰, 시간, 오류, 예산 소진, 잘림
- 호출 단계별 평균: 어느 단계의 호출이 줄거나 늘었는가
- 쌍비교 (M vs 변형, 같은 세션): 정확도 차이 Δ는 **M − 변형**이다. 양수면 부품이 정확도에 기여한다. 호출·시간 비율은 **변형 / M**이다. 1보다 작으면 부품을 빼서 줄었다는 뜻이다.


In [19]:
import subprocess
from collections import Counter

import numpy as np

ABL = {"m": "M v2 (동결)", "m_minus_router": "− 라우터", "m_minus_requery": "− 재검색",
       "m_minus_judge": "− ABSTAIN", "m_minus_compress": "− 압축"}
meta = json.load(open("results/runs/dev/r7/RUN_META.json", encoding="utf-8"))
head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
assert meta["git_commit"] == head, f"r7이 지금 커밋으로 돌지 않았다: {meta['git_commit'][:7]} != {head[:7]}"
abl = {k: load_log("r7", k) for k in ABL}

same = load_log("r3", "m").merge(abl["m"][["qid", "answer"]], on="qid", suffixes=("_r3", "_r7"))
print(f"[재현성] M v2 r3 vs r7: 답이 같은 문항 {int((same['answer_r3'] == same['answer_r7']).sum())}/{len(same)}")


def summary_row(d):
    c = d[~d["is_warmup"]]
    ans = d["answer_kind"] != "unanswerable"
    g = d.groupby("group")["correct"].sum()
    return {"정답": int(d["correct"].sum()), "GSM8K": int(g.get("gsm8k", 0)), "HotpotQA": int(g.get("hotpotqa", 0)),
            "SQuAD 답 있음": int(g.get("squad2-answerable", 0)), "SQuAD 답 없음": int(g.get("squad2-unanswerable", 0)),
            "과잉 기권": round((d["abstained"] & ans).sum() / ans.sum(), 2), "호출": round(c["n_calls"].mean(), 2),
            "입력 토큰(최대 평균)": round(c["max_input_tokens"].mean()), "시간 평균": round(c["latency_s"].mean(), 1),
            "시간 중앙": round(c["latency_s"].median(), 1), "시간 초과": int(d["timeout"].sum()),
            "오류": int(d["error"].notna().sum()), "예산 소진": int(d["budget_exhausted"].sum()),
            "잘린 호출": sum(int(x["truncated"]) for calls in d["calls"] for x in calls)}


print()
print(pd.DataFrame.from_dict({ABL[k]: summary_row(d) for k, d in abl.items()}, orient="index").to_string())

print("\n[호출 단계별 평균 (워밍업 제외)]")
steps = {}
for k, d in abl.items():
    c = d[~d["is_warmup"]]
    cnt = Counter(x["step"] for calls in c["calls"] for x in calls)
    steps[ABL[k]] = {s: round(n / len(c), 2) for s, n in sorted(cnt.items())}
print(pd.DataFrame(steps).T.fillna(0).to_string())

SEED, B = get_settings().data.seed, 10_000


def vs_m(m, v, label):
    both = m.merge(v, on="qid", suffixes=("_m", "_v"))
    cost = both[~both["is_warmup_m"] & ~both["is_warmup_v"]]
    rng = np.random.default_rng(SEED)

    def boot(fn, a, b):
        idx = rng.integers(0, len(a), size=(B, len(a)))
        lo, hi = np.percentile([fn(a[i], b[i]) for i in idx], [2.5, 97.5])
        return fn(a, b), lo, hi

    acc = boot(lambda a, b: a.mean() - b.mean(), both["correct_m"].to_numpy(float), both["correct_v"].to_numpy(float))
    calls = boot(lambda a, b: b.sum() / a.sum(), cost["n_calls_m"].to_numpy(float), cost["n_calls_v"].to_numpy(float))
    tmean = boot(lambda a, b: b.sum() / a.sum(), cost["latency_s_m"].to_numpy(float), cost["latency_s_v"].to_numpy(float))
    tmed = boot(lambda a, b: np.median(b) / np.median(a), cost["latency_s_m"].to_numpy(float), cost["latency_s_v"].to_numpy(float))
    m_only = int((both["correct_m"] & ~both["correct_v"]).sum())
    v_only = int((~both["correct_m"] & both["correct_v"]).sum())
    print(f"[{label}] M만 정답 {m_only} / 변형만 정답 {v_only} | McNemar 정확 p = {mcnemar_exact(m_only, v_only):.3f}")
    print(f"  정확도 차이 Δ (M − 변형)  {acc[0]:+.3f}  [95% {acc[1]:+.3f}, {acc[2]:+.3f}]")
    print(f"  호출 비율 (변형 / M)      ×{calls[0]:.2f}  [95% ×{calls[1]:.2f}, ×{calls[2]:.2f}]")
    print(f"  시간 비율, 평균 (변형 / M) ×{tmean[0]:.2f}  [95% ×{tmean[1]:.2f}, ×{tmean[2]:.2f}]")
    print(f"  시간 비율, 중앙값 (변형 / M) ×{tmed[0]:.2f}  [95% ×{tmed[1]:.2f}, ×{tmed[2]:.2f}]")
    if acc[1] > 0:
        verdict = "이 부품은 정확도에 기여한다 (구간이 0 위)"
    elif acc[2] < 0:
        verdict = "이 부품을 빼면 오히려 정확도가 올랐다 (구간이 0 아래)"
    else:
        verdict = "기여를 주장할 수 없음 (구간이 0을 걸침)"
    print("  =>", verdict)
    moved = both[both["correct_m"] != both["correct_v"]]
    flips = Counter((r["group_m"], "M만 정답" if r["correct_m"] else "변형만 정답") for _, r in moved.iterrows())
    print("  바뀐 문항:", dict(sorted(flips.items())) or "없음")


print()
for k in list(ABL)[1:]:
    vs_m(abl["m"], abl[k], f"M vs {ABL[k]}")
    print()


[재현성] M v2 r3 vs r7: 답이 같은 문항 58/60

           정답  GSM8K  HotpotQA  SQuAD 답 있음  SQuAD 답 없음  과잉 기권    호출  입력 토큰(최대 평균)  시간 평균  시간 중앙  시간 초과  오류  예산 소진  잘린 호출
M v2 (동결)  30     17         4           5           4   0.18  1.86           617    5.9    3.6      0   0      0      2
− 라우터      16      2         5           5           4   0.52  1.14           719    3.4    2.8      0   0      0      0
− 재검색      28     17         4           3           4   0.28  1.76           530    5.6    3.7      0   0      0      2
− ABSTAIN  31     17         4           6           4   0.14  2.27           711    7.4    6.5      0   0      0      2
− 압축       30     17         4           4           5   0.26  1.78           772    5.7    3.9      0   0      0      2

[호출 단계별 평균 (워밍업 제외)]
           finalize  judge  router
M v2 (동결)      0.12   0.76    1.02
− 라우터          0.00   1.14    0.00
− 재검색          0.12   0.66    1.02
− ABSTAIN      0.12   1.17    1.02
− 압축           0.12   0.68    1.02

[M v

## 31. 로그 남기기 (r2 ~ r7)

실행한 뒤 **노트북을 저장(Ctrl+S)** 한다. `r7`에는 5개 파이프라인 로그가 모두 들어간다.


In [20]:
dump_logs()


rounds: ['r2', 'r3', 'r4', 'r5', 'r6', 'r7'] | bytes: 92197 | sha256: 89ce91c7f8ae0c62cd24e5a324644f048f81987d7cfa70a53627bee1c020839d
A5B_LOGS_BEGIN
...
A5B_LOGS_END
